# Assignment 2.2 — streaming demo

For this demo I used the white wine csv from assignment 2.1 and trained models chunk by chunk (like data is still arriving) instead of loading everything at once.

I built two pipelines — one with a single `DecisionTreeClassifier`, one with `BaggingClassifier` — and used `StreamTrainer` to partial_fit each chunk and log accuracy. At the end I plot how accuracy changes over chunks and compare tree vs bagging.

## Setup

First cell adds `NumCompute` to the path so imports work when you open the notebook from `demo/` (I kept getting `ModuleNotFoundError` without this).

If you prefer you can `pip install -e ..` from the NumCompute folder instead and skip the path hack.

In [ ]:
import sys
from pathlib import Path

import numpy as np

%matplotlib inline


def _find_package_root():
    # notebook is usually run from NumCompute/demo/
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent / "NumCompute",
    ]
    for path in candidates:
        if (path / "numcompute").is_dir() and (path / "numcompute_stream").is_dir():
            return path.resolve()
    raise ImportError(
        "Could not find numcompute. cd to NumCompute/demo or run: pip install -e .."
    )


root = _find_package_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from numcompute.io import load_csv
from numcompute_stream.ensemble import BaggingClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.visualise import (
    compare_models,
    plot_metric_over_time,
    plot_predictions_vs_ground_truth,
)

np.set_printoptions(linewidth=120, precision=4, suppress=True)
print("using package root:", root)

## 1. Load data

Same wine file as before — 11 input columns and quality in the last column.

I turned quality into a binary label: **1** if quality >= 6 (good enough), **0** otherwise, so the tree can do classification.

In [ ]:
wine = load_csv("../assets/winequality-white.csv", dtype=np.float32, delimiter=";")
print("raw shape:", wine.shape)

X = wine[:, :11]
y = (wine[:, 11] >= 6).astype(int)

print("X shape:", X.shape)
print("class balance (0, 1):", np.bincount(y))

## 2. Split into chunks

I cut the dataset into batches of 50 rows. Each batch is one "chunk" that arrives later — that's what `partial_fit` is for in our streaming package.

In [ ]:
chunk_size = 50
chunks = [
    (X[i:i + chunk_size], y[i:i + chunk_size])
    for i in range(0, len(X), chunk_size)
]

print(f"{len(chunks)} chunks of up to {chunk_size} rows")
print("first chunk shapes:", chunks[0][0].shape, chunks[0][1].shape)

## 3. Build pipelines

Both pipelines scale features first (`StandardScaler` with partial_fit), then a model.

- `pipe_tree` — one decision tree (`max_depth=5`)
- `pipe_bagging` — bagging with 5 trees (same depth), `random_state=0` so runs are repeatable

In [ ]:
pipe_tree = Pipeline([
    ("scale", StandardScaler()),
    ("model", DecisionTreeClassifier(max_depth=5, min_samples_split=2)),
])

pipe_bagging = Pipeline([
    ("scale", StandardScaler()),
    ("model", BaggingClassifier(n_estimators=5, max_depth=5, random_state=0)),
])

## 4. Train incrementally

Loop over every chunk and call `fit_chunk` on both trainers. That partial_fits the pipeline, predicts on the same chunk, and appends to `log_` (chunk accuracy, cumulative accuracy, etc.).

I print tree vs bagging accuracy each chunk so you can see them improve as more data comes in.

In [ ]:
trainer_tree = StreamTrainer(pipe_tree)
trainer_bagging = StreamTrainer(pipe_bagging)

tree_accuracies = []
bagging_accuracies = []

for i, (X_chunk, y_chunk) in enumerate(chunks):
    trainer_tree.fit_chunk(X_chunk, y_chunk)
    trainer_bagging.fit_chunk(X_chunk, y_chunk)

    tree_acc = trainer_tree.log_[-1]["chunk_accuracy"]
    bagging_acc = trainer_bagging.log_[-1]["chunk_accuracy"]
    tree_accuracies.append(tree_acc)
    bagging_accuracies.append(bagging_acc)

    print(
        f"chunk {i + 1:2d}/{len(chunks)} | "
        f"tree={tree_acc:.3f} | bagging={bagging_acc:.3f}"
    )

## 5. Plots

Using the `visualise` module I wrote for the assignment — line plot for the tree, side-by-side comparison with bagging, and a scatter of predictions vs labels on the last chunk (bagging model).

In [ ]:
plot_metric_over_time(tree_accuracies, "Tree chunk accuracy", "Accuracy")
compare_models(tree_accuracies, bagging_accuracies, ["Tree", "Bagging"])

X_last, y_last = chunks[-1]
y_pred_last = pipe_bagging.predict(X_last)
plot_predictions_vs_ground_truth(y_last, y_pred_last)

## 6. Summary

Quick numbers at the end — how many chunks we trained on and final cumulative accuracy for both models.

In [ ]:
print(f"chunks trained: {len(chunks)}")
print(f"tree cumulative accuracy: {trainer_tree.log_[-1]['cumulative_accuracy']:.3f}")
print(f"bagging cumulative accuracy: {trainer_bagging.log_[-1]['cumulative_accuracy']:.3f}")
print(f"tree scaler mean (first 3 features): {pipe_tree.named_steps['scale'].mean_[:3]}")